In [1]:
import pandas as pd
import zipfile
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import CategoricalNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,roc_auc_score,precision_score,recall_score,f1_score, matthews_corrcoef

In [2]:
def get_predictions(model, y_test):
    """ Helper function to predict outcome on test data"""    
    # Generate predictions
    y_pred = model.predict(X_test)

    # Extract probabilities for AUC calculation 
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.decision_function(X_test)

    return y_pred, y_prob

In [3]:
def evaluate_model(y_test, y_pred, y_prob):    
    """ Helper function to calculate the metrics"""    
    # Calculate the 6 specific metrics
    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC Score": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "MCC Score": matthews_corrcoef(y_test, y_pred)
    }
    
    return metrics

In [4]:
def print_metrics(metrics, model_name="Model"):
    """ Helper function to print the metrics""" 
    # Print formatted output
    print(f"\n=== {model_name} Evaluation Metrics ===")
    for metric_name, value in metrics.items():
        print(f"{metric_name:<12}: {value:.4f}")
    print("=" * (len(model_name) + 25))

In [5]:
# Path to downloaded zip file
zip_path = "mushroom.zip" 

# Extract and read the data file from the ZIP
data_filename = "agaricus-lepiota.data"

columns = [
    "class", "cap-shape", "cap-surface", "cap-color", "bruises", "odor",
    "gill-attachment", "gill-spacing", "gill-size", "gill-color",
    "stalk-shape", "stalk-root", "stalk-surface-above-ring",
    "stalk-surface-below-ring", "stalk-color-above-ring",
    "stalk-color-below-ring", "veil-type", "veil-color", "ring-number",
    "ring-type", "spore-print-color", "population", "habitat"
]

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open(data_filename) as f:
        df = pd.read_csv(f, header=None, names=columns)

# Print number of rows and columns
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Number of rows: 8124
Number of columns: 23


In [6]:
# Separate features (X) and target label (y)
X = df.drop("class", axis=1)
y = df["class"]

# Split the data into train and test
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
print("--- Column Data Types ---")
print(X_tr.dtypes)

# 2. Check for standard missing values (Will output all 0s)
print("\n--- Standard Missing Values (NaN/Null) ---")
print(X_tr.isnull().sum())

--- Column Data Types ---
cap-shape                   object
cap-surface                 object
cap-color                   object
bruises                     object
odor                        object
gill-attachment             object
gill-spacing                object
gill-size                   object
gill-color                  object
stalk-shape                 object
stalk-root                  object
stalk-surface-above-ring    object
stalk-surface-below-ring    object
stalk-color-above-ring      object
stalk-color-below-ring      object
veil-type                   object
veil-color                  object
ring-number                 object
ring-type                   object
spore-print-color           object
population                  object
habitat                     object
dtype: object

--- Standard Missing Values (NaN/Null) ---
cap-shape                   0
cap-surface                 0
cap-color                   0
bruises                     0
odor                      

In [8]:
print("\n--- Hidden Missing Values ('?') ---")
hidden_missing = (X_tr == '?').sum()
print(hidden_missing[hidden_missing > 0])


--- Hidden Missing Values ('?') ---
stalk-root    2007
dtype: int64


In [9]:
most_frequent = X_tr[X_tr['stalk-root'] != '?']['stalk-root'].mode()[0]

# Replace all '?' values with that most frequent value from training data
X_tr['stalk-root'] = X_tr['stalk-root'].replace('?', most_frequent)
X_te['stalk-root'] = X_te['stalk-root'].replace('?', most_frequent)

In [10]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
# Fit and Transform
X_train_encoded = encoder.fit_transform(X_tr)
X_train = pd.DataFrame(X_train_encoded, columns=X_tr.columns)

In [11]:
target_map = {'e': 0, 'p': 1}
y_train = y_tr.map(target_map).astype(int)

In [12]:
X_test_encoded = encoder.transform(X_te)
X_test = pd.DataFrame(X_test_encoded, columns=X_te.columns)
y_test = y_te.map(target_map).astype(int)

In [13]:
# Initialize the Logistic Regression Model
lr = LogisticRegression(max_iter=500)

# Fit the model
lr.fit(X_train, y_train)

# Generate predictions
y_pred, y_prob = get_predictions(lr, y_test)

# Evaluvate model anb print metrics
metrics = evaluate_model(y_test, y_pred, y_prob)
print_metrics(metrics, model_name="Logistic Regression")


=== Logistic Regression Evaluation Metrics ===
Accuracy    : 0.9532
AUC Score   : 0.9834
Precision   : 0.9480
Recall      : 0.9552
F1 Score    : 0.9516
MCC Score   : 0.9064


In [14]:
# Initialize the Decision Tree Classifier
dt = DecisionTreeClassifier(random_state=42)

# Fit the model
dt.fit(X_train, y_train)
    
# Generate predictions
y_pred, y_prob = get_predictions(dt, y_test)

# Evaluvate model anb print metrics
metrics = evaluate_model(y_test, y_pred, y_prob)
print_metrics(metrics, model_name="Decision Tree")


=== Decision Tree Evaluation Metrics ===
Accuracy    : 1.0000
AUC Score   : 1.0000
Precision   : 1.0000
Recall      : 1.0000
F1 Score    : 1.0000
MCC Score   : 1.0000


In [15]:
# Initialize the KNN model
knn = KNeighborsClassifier(n_neighbors=3)

# Fit the model
knn.fit(X_train, y_train)

# Generate predictions
y_pred, y_prob = get_predictions(knn, y_test)

# Evaluvate model anb print metrics
metrics = evaluate_model(y_test, y_pred, y_prob)
print_metrics(metrics, model_name="KNN")


=== KNN Evaluation Metrics ===
Accuracy    : 0.9982
AUC Score   : 1.0000
Precision   : 0.9962
Recall      : 1.0000
F1 Score    : 0.9981
MCC Score   : 0.9963


In [16]:
# Initialize the Categorical Naive bayes model as all our features are categorical
nb = CategoricalNB()

# Fit the model
nb.fit(X_train, y_train)

# Generate predictions
y_pred, y_prob = get_predictions(nb, y_test)

# Evaluvate model anb print metrics
metrics = evaluate_model(y_test, y_pred, y_prob)
print_metrics(metrics, model_name="Naive Bayes")


=== Naive Bayes Evaluation Metrics ===
Accuracy    : 0.9508
AUC Score   : 0.9975
Precision   : 0.9875
Recall      : 0.9092
F1 Score    : 0.9467
MCC Score   : 0.9038


In [17]:
# Initialize the Random Forest model
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Fit the model
rf.fit(X_train, y_train)

# Generate predictions
y_pred, y_prob = get_predictions(rf, y_test)

# Evaluvate model anb print metrics
metrics = evaluate_model(y_test, y_pred, y_prob)
print_metrics(metrics, model_name="Randon Forest")


=== Randon Forest Evaluation Metrics ===
Accuracy    : 1.0000
AUC Score   : 1.0000
Precision   : 1.0000
Recall      : 1.0000
F1 Score    : 1.0000
MCC Score   : 1.0000


In [18]:
import joblib

# Save all the models
joblib.dump(lr, "./model/lr.joblib")
joblib.dump(dt, "./model/dt.joblib")
joblib.dump(knn, "./model/knn.joblib")
joblib.dump(nb, "./model/nb.joblib")
joblib.dump(rf, "./model/rf.joblib")
joblib.dump(encoder, "./model/encoder.joblib")

['./model/encoder.joblib']

In [19]:
# Save the test data
test_df = X_te.copy()
test_df['class'] = y_te

# Save to a CSV file
test_df.to_csv("mushrooms_test.csv", index=False)